In [0]:
landing_table=dbutils.widgets.get("landing_table")
landing_flattened_table=dbutils.widgets.get("landing_flattened_table")
raw_table=dbutils.widgets.get("raw_table")
volume_path=dbutils.widgets.get("volume_path")

In [0]:
try:
    spark.sql(f"""
        WITH latest_src AS (
            SELECT *
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY TransID, Attachments_TranAttachmentsID, DataRecord, RuleOverrides, Services_ServiceID, Services_LineNo
                        ORDER BY _load_timestamp DESC
                    ) AS rn
                FROM {landing_flattened_table}
            ) t
            WHERE rn = 1
        )

        MERGE INTO {raw_table} tgt
        USING latest_src src
        ON tgt.TransID = src.TransID
        AND COALESCE(tgt.Attachments_TranAttachmentsID,'') = COALESCE(src.Attachments_TranAttachmentsID,'')
        AND COALESCE(tgt.DataRecord,'') = COALESCE(src.DataRecord,'')
        AND COALESCE(tgt.RuleOverrides,'') = COALESCE(src.RuleOverrides,'')
        AND COALESCE(tgt.Services_ServiceID,-1) = COALESCE(src.Services_ServiceID,-1)
        AND COALESCE(tgt.Services_LineNo,-1) = COALESCE(src.Services_LineNo,-1)
        AND tgt._modified_ts < src._load_timestamp
        AND tgt._active_flag = 1

        WHEN MATCHED THEN
            UPDATE SET
                tgt._active_flag = 0,
                tgt._modified_ts = src._load_timestamp
    """)
except Exception as e:
    print(f"Error updating {raw_table}: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        WITH latest_src AS (
            SELECT *
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY TransID, Attachments_TranAttachmentsID, DataRecord, RuleOverrides, Services_ServiceID, Services_LineNo
                        ORDER BY _load_timestamp DESC
                    ) AS rn
                FROM {landing_flattened_table}
            ) t
            WHERE rn = 1
        )

        MERGE INTO {raw_table} tgt
        USING latest_src src
        ON tgt.TransID = src.TransID
        AND COALESCE(tgt.Attachments_TranAttachmentsID,'') = COALESCE(src.Attachments_TranAttachmentsID,'')
        AND COALESCE(tgt.DataRecord,'') = COALESCE(src.DataRecord,'')
        AND COALESCE(tgt.RuleOverrides,'') = COALESCE(src.RuleOverrides,'')
        AND COALESCE(tgt.Services_ServiceID,-1) = COALESCE(src.Services_ServiceID,-1)
        AND COALESCE(tgt.Services_LineNo,-1) = COALESCE(src.Services_LineNo,-1)
        AND tgt._created_ts = src._load_timestamp

        WHEN NOT MATCHED THEN
        INSERT (
            AcceptAssignment,
            AssignedToID,
            Attachments_AttachmentTransID,
            Attachments_ControlNumber,
            Attachments_Description,
            Attachments_FileID,
            Attachments_Pending,
            Attachments_ReportType,
            Attachments_TranAttachmentsID,
            Attachments_TransmissionCode,
            AttendingFirst,
            AttendingLast,
            AttendingProvID_NPI,
            BillType,
            BillingAddr1,
            BillingCity,
            BillingEntType,
            BillingName,
            BillingProvID_CommercialNo,
            BillingProvID_EIN,
            BillingProvID_NPI,
            BillingProvID_Taxonomy,
            BillingState,
            BillingZip,
            Box10d,
            ClaimCharge,
            ClaimDate_Accident,
            ClaimDate_OnsetCurrent,
            ClaimID_MammoCert,
            ClaimID_MedicalRecNo,
            ClaimPaidInd,
            ClaimPayment,
            ClaimType,
            Contract_Code,
            CreateDate,
            CreateMode,
            DataRecord,
            DestInsNo,
            DiagnosisCodes10_Other1,
            DiagnosisCodes10_Other2,
            DiagnosisCodes10_Other3,
            DiagnosisCodes10_Other4,
            DiagnosisCodes10_Other5,
            DiagnosisCodes10_Other6,
            DiagnosisCodes10_Other7,
            DiagnosisCodes10_Other8,
            DiagnosisCodes10_Principal,
            DiagnosisVersion,
            EditByID,
            EditDate,
            ImportBatchID,
            InsuranceLastID,
            Insurances_1_AssignBenefits,
            Insurances_1_ClaimIndCode,
            Insurances_1_GroupNumber,
            Insurances_1_InsuranceID,
            Insurances_1_PatientRelateCode,
            Insurances_1_PayerAddr1,
            Insurances_1_PayerCity,
            Insurances_1_PayerICN,
            Insurances_1_PayerID_ClmOfficeNo,
            Insurances_1_PayerID_ID,
            Insurances_1_PayerID_Local,
            Insurances_1_PayerID_LocalIndCode,
            Insurances_1_PayerName,
            Insurances_1_PayerPartnerID,
            Insurances_1_PayerSeqCode,
            Insurances_1_PayerSeqNo,
            Insurances_1_PayerState,
            Insurances_1_PayerZip,
            Insurances_1_PriorAuthNo,
            Insurances_1_PropertyClmNo,
            Insurances_1_ReleaseInfoCode,
            Insurances_1_SubscriberAddr1,
            Insurances_1_SubscriberAddr2,
            Insurances_1_SubscriberCity,
            Insurances_1_SubscriberDOB,
            Insurances_1_SubscriberFirst,
            Insurances_1_SubscriberGender,
            Insurances_1_SubscriberID_MemberID,
            Insurances_1_SubscriberLast,
            Insurances_1_SubscriberMiddle,
            Insurances_1_SubscriberState,
            Insurances_1_SubscriberZip,
            LastUpdated,
            LastUpdatedUserId,
            LastUserUpdatedDate,
            MediaCode,
            NoteLastID,
            Notes,
            ParentID,
            PatID,
            PatientAddr1,
            PatientAddr2,
            PatientCity,
            PatientCtlNo,
            PatientDOB,
            PatientFirst,
            PatientGender,
            PatientLast,
            PatientMiddle,
            PatientState,
            PatientZip,
            PayToAddr1,
            PayToCity,
            PayToName,
            PayToProvID_NPI,
            PayToProvID_Taxonomy,
            PayToState,
            PayToZip,
            PayerMatchID,
            PayerPartnerID,
            PaymentDate,
            PrintMail,
            ProcedureVersion,
            ProvMatchID,
            ProvPartnerID,
            ReferringFirst,
            ReferringLast,
            ReferringMiddle,
            ReferringProvID_NPI,
            ReimbursementAmount,
            RelatedEmployment,
            ReleaseToDde,
            RemitClaimID,
            RenderingEntType,
            RenderingFirst,
            RenderingLast,
            RenderingMiddle,
            RenderingProvID_NPI,
            RenderingProvID_Taxonomy,
            RenderingSuffix,
            RuleOverrides,
            ServiceAddr1,
            ServiceAddr2,
            ServiceCity,
            ServiceLastID,
            ServiceName,
            ServiceState,
            ServiceZip,
            ServiceLineNo,
            Services_ThroughDate,
            Services_DiagnosisPointer,
            Services_OrderingMiddle,
            Services_Charge,
            Services_OrderingProvID_NPI,
            Services_RenderingProvID_NPI,
            Services_Units,
            Services_HCPC,
            Services_ServiceID,
            Services_OrderingSuffix,
            Services_PlaceService,
            Services_UnderpaymentAmount,
            Services_OrderingFirst,
            Services_ReferringSuffix,
            Services_OrderingLast,
            Services_RenderingLast,
            Services_FromDate,
            Services_LineNo,
            SignatureOnFile,
            SourceFileID,
            StatementEnd,
            StatementStart,
            TransID,
            TransStatus,
            TransType,
            TransmitDate,
            UnderpaymentAmount,
            adj_icn,
            _file_name,
            _created_ts,
            _modified_ts,
            _active_flag
        )
        VALUES (
            src.AcceptAssignment,
            src.AssignedToID,
            src.Attachments_AttachmentTransID,
            src.Attachments_ControlNumber,
            src.Attachments_Description,
            src.Attachments_FileID,
            src.Attachments_Pending,
            src.Attachments_ReportType,
            src.Attachments_TranAttachmentsID,
            src.Attachments_TransmissionCode,
            src.AttendingFirst,
            src.AttendingLast,
            src.AttendingProvID_NPI,
            src.BillType,
            src.BillingAddr1,
            src.BillingCity,
            src.BillingEntType,
            src.BillingName,
            src.BillingProvID_CommercialNo,
            src.BillingProvID_EIN,
            src.BillingProvID_NPI,
            src.BillingProvID_Taxonomy,
            src.BillingState,
            src.BillingZip,
            src.Box10d,
            src.ClaimCharge,
            src.ClaimDate_Accident,
            src.ClaimDate_OnsetCurrent,
            src.ClaimID_MammoCert,
            src.ClaimID_MedicalRecNo,
            src.ClaimPaidInd,
            src.ClaimPayment,
            src.ClaimType,
            src.Contract_Code,
            src.CreateDate,
            src.CreateMode,
            src.DataRecord,
            src.DestInsNo,
            src.DiagnosisCodes10_Other1,
            src.DiagnosisCodes10_Other2,
            src.DiagnosisCodes10_Other3,
            src.DiagnosisCodes10_Other4,
            src.DiagnosisCodes10_Other5,
            src.DiagnosisCodes10_Other6,
            src.DiagnosisCodes10_Other7,
            src.DiagnosisCodes10_Other8,
            src.DiagnosisCodes10_Principal,
            src.DiagnosisVersion,
            src.EditByID,
            src.EditDate,
            src.ImportBatchID,
            src.InsuranceLastID,
            src.Insurances_1_AssignBenefits,
            src.Insurances_1_ClaimIndCode,
            src.Insurances_1_GroupNumber,
            src.Insurances_1_InsuranceID,
            src.Insurances_1_PatientRelateCode,
            src.Insurances_1_PayerAddr1,
            src.Insurances_1_PayerCity,
            src.Insurances_1_PayerICN,
            src.Insurances_1_PayerID_ClmOfficeNo,
            src.Insurances_1_PayerID_ID,
            src.Insurances_1_PayerID_Local,
            src.Insurances_1_PayerID_LocalIndCode,
            src.Insurances_1_PayerName,
            src.Insurances_1_PayerPartnerID,
            src.Insurances_1_PayerSeqCode,
            src.Insurances_1_PayerSeqNo,
            src.Insurances_1_PayerState,
            src.Insurances_1_PayerZip,
            src.Insurances_1_PriorAuthNo,
            src.Insurances_1_PropertyClmNo,
            src.Insurances_1_ReleaseInfoCode,
            src.Insurances_1_SubscriberAddr1,
            src.Insurances_1_SubscriberAddr2,
            src.Insurances_1_SubscriberCity,
            src.Insurances_1_SubscriberDOB,
            src.Insurances_1_SubscriberFirst,
            src.Insurances_1_SubscriberGender,
            src.Insurances_1_SubscriberID_MemberID,
            src.Insurances_1_SubscriberLast,
            src.Insurances_1_SubscriberMiddle,
            src.Insurances_1_SubscriberState,
            src.Insurances_1_SubscriberZip,
            src.LastUpdated,
            src.LastUpdatedUserId,
            src.LastUserUpdatedDate,
            src.MediaCode,
            src.NoteLastID,
            src.Notes,
            src.ParentID,
            src.PatID,
            src.PatientAddr1,
            src.PatientAddr2,
            src.PatientCity,
            src.PatientCtlNo,
            src.PatientDOB,
            src.PatientFirst,
            src.PatientGender,
            src.PatientLast,
            src.PatientMiddle,
            src.PatientState,
            src.PatientZip,
            src.PayToAddr1,
            src.PayToCity,
            src.PayToName,
            src.PayToProvID_NPI,
            src.PayToProvID_Taxonomy,
            src.PayToState,
            src.PayToZip,
            src.PayerMatchID,
            src.PayerPartnerID,
            src.PaymentDate,
            src.PrintMail,
            src.ProcedureVersion,
            src.ProvMatchID,
            src.ProvPartnerID,
            src.ReferringFirst,
            src.ReferringLast,
            src.ReferringMiddle,
            src.ReferringProvID_NPI,
            src.ReimbursementAmount,
            src.RelatedEmployment,
            src.ReleaseToDde,
            src.RemitClaimID,
            src.RenderingEntType,
            src.RenderingFirst,
            src.RenderingLast,
            src.RenderingMiddle,
            src.RenderingProvID_NPI,
            src.RenderingProvID_Taxonomy,
            src.RenderingSuffix,
            src.RuleOverrides,
            src.ServiceAddr1,
            src.ServiceAddr2,
            src.ServiceCity,
            src.ServiceLastID,
            src.ServiceName,
            src.ServiceState,
            src.ServiceZip,
            src.ServiceLineNo,
            src.Services_ThroughDate,
            src.Services_DiagnosisPointer,
            src.Services_OrderingMiddle,
            src.Services_Charge,
            src.Services_OrderingProvID_NPI,
            src.Services_RenderingProvID_NPI,
            src.Services_Units,
            src.Services_HCPC,
            src.Services_ServiceID,
            src.Services_OrderingSuffix,
            src.Services_PlaceService,
            src.Services_UnderpaymentAmount,
            src.Services_OrderingFirst,
            src.Services_ReferringSuffix,
            src.Services_OrderingLast,
            src.Services_RenderingLast,
            src.Services_FromDate,
            src.Services_LineNo,
            src.SignatureOnFile,
            src.SourceFileID,
            src.StatementEnd,
            src.StatementStart,
            src.TransID,
            src.TransStatus,
            src.TransType,
            src.TransmitDate,
            src.UnderpaymentAmount,
            src.adj_icn,
            src._file_name,
            src._load_timestamp,
            src._load_timestamp,
            1
        );

    """)
except Exception as e:
    print(f"Error inserting into {raw_table}: {e}")
    raise